##### Ambition regression with industry presence as a dummy

Step 1: Merge climactor boundaries with climactor ambition dataset <br>
Step 2: Get idustry presence and capcaity in cities <br>
Step 3 Merge country ambition with climactor ambition <br>
Step 4: Prepare data for regression <br>
Step 5: Run the regression <br>

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import os
import datetime
from shapely.geometry import shape
import json
from shapely import wkt

CLIMACTOR = r"C:\Users\Marti183\CAMDA_NSA\problem-shifting\data\scripts\ClimActor_ambition.2026_07_16.csv"
PBL = r"C:\Users\Marti183\CAMDA_NSA\problem-shifting\data\scripts\pbl_annualised_ambition_2026_07_14.csv"
CAT = r"C:\Users\Marti183\CAMDA_NSA\problem-shifting\data\scripts\CAT_annualised_ambition.2026_07_14.csv"

STEEL_PLANTS = r"C:\Users\Marti183\CAMDA_NSA\problem-shifting\data\data_raw\steel_gdf_2026_05_18.csv"
CEMENT_PLANTS = r"C:\Users\Marti183\CAMDA_NSA\problem-shifting\data\data_raw\cement_gdf_2026_07_07.csv"
POWER_PLANTS = r"C:\Users\Marti183\CAMDA_NSA\problem-shifting\data\data_raw\cpp_finest_ADM_merged_by_gem_ids_CPP_2026.csv"

CRS="EPSG:4326"

In [19]:
# read files

def read_files(files):
    file = pd.read_csv(files)

    return file

In [20]:
climactor = read_files(CLIMACTOR)
pbl = read_files(PBL)
cat = read_files(CAT)

In [21]:
#Climactor is the only one that contains cities, so first i need to match the climactor id with the name of the city and its boundaries

# This function is to load climactor boundaries
def load_climactor_boundaries():
    path = r"C:\Users\Marti183\CAMDA_NSA\Gdam_ClimActor_data"

    files = os.listdir(path)
    data = []
    
    for i in range(1,6):
        gdfs=[]
        #first split the files per ADM (otherwise the code breaks cuz the data to manage is too large)
        content = [f for f in files if f.split("_")[1] == f"{i}"]
        
        for value in content:
            #turn each file inot a gdf
            file = os.path.join(path, value)
            #With the newest boundaries shared by Yuetong Im getting out of memory errors
            #The files were just too big >1GB
            gdf = gpd.GeoDataFrame.from_file(file, crs=CRS)
            gdfs.append(gdf)

        gdf_final = pd.concat(gdfs, ignore_index=True, axis=0)

        #clean the gdf to only include relevant columns
        columns = [f"GID_{i}"]
        columns.append("geometry")
        gdf_final = gdf_final[columns]
        gdf_final = gdf_final.drop_duplicates().reset_index()
        gdf_final = gdf_final.rename(columns={f"GID_{i}": "GID"})
        data.append(gdf_final)

    #concat all the gdf_finals so that I have only one single file with all boundaries
    
    final = pd.concat(data, ignore_index=True, axis=0).reset_index()
    final = final[["GID", "geometry"]]
    return final

climactor_boundaries = load_climactor_boundaries()

In [22]:
climactor

,Unnamed: 0,iso,entity_type,climactor_id,baseline_value,baseline_year,inv_year,target_year,target_value,target_record,GDAM_id,target_year_emission,annualised_ambition_from_baseline,annualised_ambition,2019_emissions_for_total_ambition,2030_emissions,absolute_ambition
0,6,ESP,City,00752000db04c515,11423.000,2005,2014.0,2030,40.00,00752000db04c515_2030,ESP.09.08.08222,6853.80000,0.013550,0.012225,9.436959e+03,6.853800e+03,2583.159046
1,7,ITA,City,0083ef4ed71b869e,7739.785,2011,2011.0,2030,40.00,0083ef4ed71b869e_2030,ITA.19.86.003,4643.87100,0.017867,0.017867,6.700266e+03,4.643871e+03,2056.394823
2,10,BEL,City,00887dae5af7b824,48759.200,2011,2020.0,2030,40.00,00887dae5af7b824_2030,BEL.2.1.3.22_1,29255.52000,0.017867,0.030747,4.221042e+04,2.925552e+04,12954.903325
3,13,ITA,City,00b3f9e9c6778784,8965.400,2011,2011.0,2030,40.00,00b3f9e9c6778784_2030,ITA.19.84.007,5379.24000,0.017867,0.017867,7.761270e+03,5.379240e+03,2382.030269
4,17,ITA,City,00db591fd214bead,7876.169,2011,2011.0,2030,40.00,00db591fd214bead_2030,ITA.19.83.088,4725.70140,0.017867,0.017867,6.818332e+03,4.725701e+03,2092.630888
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2382,9387,USA,City,ff4e338346fcdebb,2597305.150,2019,2019.0,2050,100.00,ff4e338346fcdebb_2050,USA.16.77.1600000US1921000,0.00000,0.022611,0.022611,2.597305e+06,2.019588e+06,577717.230931
2383,9393,ITA,City,ff8d794fe84c1e9d,22806.700,2011,2011.0,2050,40.42,ff8d794fe84c1e9d_2050,ITA.19.87.047,13588.23186,0.008742,0.008742,2.125960e+04,1.930223e+04,1957.372294
2384,9400,BEL,City,ff904d7013b2688c,75769.000,2011,2020.0,2030,40.00,ff904d7013b2688c_2030,BEL.2.2.3.1_1,45461.40000,0.017867,0.028499,6.559258e+04,4.546140e+04,20131.176682
2385,9403,ITA,City,ffa60dd141d5c943,83747.800,2010,2010.0,2030,40.00,ffa60dd141d5c943_2030,ITA.15.63.073,50248.68000,0.016966,0.016966,7.179440e+04,5.024868e+04,21545.717128


In [23]:
#This function is to merge climactor annualised ambition dataset with climactor boundaries (because the pbl and CAT are only for countries)
def add_boundaries_to_ambition(ambition=climactor, boundaries=climactor_boundaries):
    new_ambition = pd.merge(ambition, boundaries, how="left", left_on="GDAM_id", right_on="GID")
    new_ambition = new_ambition.drop(columns="GID")

    return new_ambition

In [24]:
new_ambition = add_boundaries_to_ambition()

##### The next step is to perform a spatial join between industries datasets (power, cement and steel) and climactor annualised ambition

In [25]:
#First I need to load the industry datasets
    
power = read_files(POWER_PLANTS)
cement = read_files(CEMENT_PLANTS)
steel = read_files(STEEL_PLANTS)

In [26]:
#To perform a spatial join both datsets must be geopandas

def to_gdf(data):
    #isinstance(data["geometry"], str) is always false because data["geometry"] is a pandas SERIES its the elements inside what are strings 
    #what I can do is test the first element
    if ".geo" in data.columns and isinstance(data[".geo"].iloc[0], str):
        #.geo column contains JSON text (STRING).
        # json.loads() parses the text into a Python dictionary.
        # shape() converts that dictionary into a Shapely geometry object. (because shape expects a dict not a string that why I need the json.loads method)
        #so shape works with json type which is: type: blabla 
        #if its directly " POINT (89.5 , 43.5)" and its a string then is in wtk so I have to use wkt.loads and dont need shpe at all
        data["geometry"] = data[".geo"].apply(lambda x: shape(json.loads(x)) if isinstance(x, str) else shape(x))

        gdf = gpd.GeoDataFrame(data, geometry="geometry", crs=CRS)

    elif "geometry" in data.columns and isinstance(data["geometry"].iloc[0], str):
        data["geometry"] = data["geometry"].apply(lambda x: wkt.loads(x) if isinstance(x, str) else shape(x))
        gdf = gpd.GeoDataFrame(data, geometry="geometry", crs=CRS)

    elif "geometry" in data.columns:
        gdf = gpd.GeoDataFrame(data, geometry="geometry", crs=CRS)

    else:
        raise ValueError("No geometry column found")

    return gdf

power_gdf = to_gdf(power)
cement_gdf = to_gdf(cement)
steel_gdf = to_gdf(steel)
new_ambition_gdf = to_gdf(new_ambition)
    

In [27]:

#I will need the sum of the capacity for the second regression 
#For cement and steel I need to sum the clinker+cement capacities and the iron+steel capacities, for the power plant i dont theres only one column: Capacity (MW)
def add_capacities(industry, industry_string):
    if "cement" in industry_string  or "steel" in industry_string:
        capacity_columns = [col for col in industry.columns if "capacity" in col.lower()]
        industry[f"{industry_string}_capacity"] = industry[capacity_columns].sum(axis=1)
        #clean the dataset for spatial join only include relevant columns
        industry = industry[[f"{industry_string}_capacity", "geometry"]]
    
    else:
        industry = industry[["Capacity (MW)", "geometry"]]
        industry =industry.rename(columns={"Capacity (MW)": f"{industry_string}_capacity"})

    return industry



#This function performs a spatial join between climactor ambition and the industries

def industry_presence(industry, industry_string, ambition=new_ambition_gdf):
    industry_data = add_capacities(industry, industry_string)
    ##when I use3 within it menas left geometries are inside right geometries so If I use ip = gpd.sjoin(ambition, industry_data, how="left", predicate="within") Im saying are cities boundaries inside industry points? The opposite of what I want
    ip = gpd.sjoin(industry_data, ambition, how="right", predicate="within")
  
    summary = (
    ip.assign(present= ip[f"{industry_string}_capacity"].notna().astype(int))
      .groupby("GDAM_id", as_index=False)
      .agg(
          presence=("present", "max"),
          capacity=(f"{industry_string}_capacity", "sum")
      ))
    
    summary = summary.rename(columns={"presence": f"{industry_string}_presence", "capacity":f"{industry_string}_capacity"})
    return summary


In [28]:
#   This function merges back the capacity and presence columns to the ambition dataset

def adding_results_to_ambition(summary, ambition):
    merged_df = pd.merge(ambition, summary, how="left", on="GDAM_id")
    return merged_df

In [29]:
power_summary = industry_presence(power_gdf, "power")
steel_summary = industry_presence(steel_gdf, "steel")
cement_summary = industry_presence(cement_gdf, "cement")


In [30]:
new_ambition_gdf=adding_results_to_ambition(power_summary,new_ambition_gdf)
new_ambition_gdf=adding_results_to_ambition(steel_summary, new_ambition_gdf)
new_ambition_gdf=adding_results_to_ambition(cement_summary, new_ambition_gdf)


#### The next step is to add the country ambition to the climactor dataset. In the climactor dataset there are no country ambitions so I have to merge the PBL or the CAT datasets 
##### I will calculate two different ambitions with each country level data (PBL or CAT) for NDC Unconditional sceanrio

In [108]:
#This function is to add the country ambition to the climactor ambition dataset

def add_PBL_country_ambition(pbl_ambition=pbl, climactor=new_ambition_gdf):
   
    data_regression = pd.merge(pbl_ambition, climactor, how="right", left_on="ISO", right_on="iso")

    #Add one binary variable to check whther a country has ambition or not
    data_regression["country_ambition"] = np.where(data_regression["annualised_geometric_ambition"].isna(), 0, 1)

    #Add the ISO of the country for those countries with no ambition
    data_regression["ISO"] = data_regression["GDAM_id"].str.split(".").str[0]

    #Fill np.nan values with 0
    data_regression[["annualised_geometric_ambition", "total_ambition"]] = data_regression[["annualised_geometric_ambition", "total_ambition"]].fillna(0)
    return data_regression


In [109]:
def add_CAT_country_ambition(cat_ambition = cat, climactor= new_ambition_gdf):
    data_regression = pd.merge(cat_ambition, climactor, how="right", left_on="Country", right_on="iso")
    #Add one binary variable to check whther a country has ambition or not
    data_regression["country_ambition"] = np.where(data_regression["NDC Unconditional_total_ambition"].isna(), 0, 1)

    #Add the ISO of the country for thsoe countries with no ambition
    data_regression["iso"] = data_regression["GDAM_id"].str.split(".").str[0]
    #Fill np.nan values with 0 because inplace is less safe
    data_regression[["NDC Unconditional_annualised_geometric_ambition", "NDC Unconditional_total_ambition"]] = data_regression[["NDC Unconditional_annualised_geometric_ambition", "NDC Unconditional_total_ambition"]].fillna(0)

    return data_regression


Regression analysis

Data preparation for two regressions: annualised mabition + industry presence and total ambition + industry capacity

In [110]:
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm

In [215]:
#Data cleaning to only inlcude relevant columns for regression type I
def pbl_type_I_reg():
    data = add_PBL_country_ambition()
    data = data[["ISO", "country_ambition", "annualised_geometric_ambition", "GDAM_id", "annualised_ambition", "power_presence", "steel_presence", "cement_presence"]]
    data.to_csv("pbl_type_I_reg.csv")
    #data = data[data["country_ambition"] == 1]


    X = data[["country_ambition",
        "annualised_geometric_ambition", 
        "power_presence", "steel_presence", "cement_presence" ]]
    Y = data["annualised_ambition"]

    # X = sm.add_constant(X)
    model = sm.OLS(Y, X).fit()
    print(data[["power_presence", "steel_presence", "cement_presence", "country_ambition", "annualised_geometric_ambition"]].corr())
    

    return print(model.summary())

In [216]:
def cat_type_I_reg():
    data = add_CAT_country_ambition()
    data = data[["iso", "country_ambition", "NDC Unconditional_annualised_geometric_ambition", "GDAM_id", "annualised_ambition", "power_presence", "steel_presence", "cement_presence"]]
    data.to_csv("cat_type_I_reg.csv")
    #data = data[data["country_ambition"] == 1]

    X = data[["country_ambition",
         "NDC Unconditional_annualised_geometric_ambition", 
         "power_presence", "steel_presence", "cement_presence" ]]
    Y = data["annualised_ambition"]

    # X = sm.add_constant(X)
    model = sm.OLS(Y, X).fit()
    print(data[["power_presence", "steel_presence", "cement_presence"]].corr())

    return print(model.summary())

In [217]:
def pbl_type_II_reg():
     data = add_PBL_country_ambition()
     data = data[["ISO", "country_ambition", "total_ambition", "GDAM_id", "absolute_ambition", "power_capacity", "steel_capacity", "cement_capacity"]]
     data.to_csv("pbl_type_II_reg.csv")
#      data = data[data["country_ambition"] == 1]


     X = data[["country_ambition",
           "total_ambition", "power_capacity", "steel_capacity", "cement_capacity" ]]
     Y = data["absolute_ambition"]

     X = sm.add_constant(X)
     model = sm.OLS(Y, X).fit()
     
     return print(model.summary())

In [218]:
def cat_type_II_reg():
    data = add_CAT_country_ambition()
    data = data[["iso", "country_ambition", "NDC Unconditional_total_ambition", "GDAM_id", "absolute_ambition", "power_capacity", "steel_capacity", "cement_capacity"]]
    data.to_csv("cat_type_II_reg.csv")
#   data = data[data["country_ambition"] == 1]

    X = data[["country_ambition",
         "NDC Unconditional_total_ambition", "power_capacity", "steel_capacity", "cement_capacity" ]]
    Y = data["absolute_ambition"]

    X = sm.add_constant(X)
    model = sm.OLS(Y, X).fit()
    print(data[["power_capacity", "steel_capacity", "cement_capacity", "country_ambition", "NDC Unconditional_total_ambition"]].corr())

    return print(model.summary())

In [219]:
pbl_type_I_reg = pbl_type_I_reg()
pbl_type_II_reg = pbl_type_II_reg()
cat_type_I_reg = cat_type_I_reg()
cat_type_II_reg= cat_type_II_reg()



                               power_presence  steel_presence  \
power_presence                       1.000000        0.223444   
steel_presence                       0.223444        1.000000   
cement_presence                      0.184849        0.095747   
country_ambition                     0.162110        0.107526   
annualised_geometric_ambition        0.123202        0.100147   

                               cement_presence  country_ambition  \
power_presence                        0.184849          0.162110   
steel_presence                        0.095747          0.107526   
cement_presence                       1.000000          0.081744   
country_ambition                      0.081744          1.000000   
annualised_geometric_ambition         0.083859          0.900779   

                               annualised_geometric_ambition  
power_presence                                      0.123202  
steel_presence                                      0.100147  
cement_pres

In [ ]:
cat_type_I_reg

In [ ]:

a = pd.read_csv("cat_type_I_reg.csv")
a["power_presence"].sum()
a["steel_presence"].sum()
a["cement_presence"].sum()
b = a[a["country_ambition"] == 1]
b[["power_presence", "steel_presence", "cement_presence"]].sum(axis=1).sum()

np.int64(44)

In [142]:
new_ambition_gdf[["power_presence", "power_capacity"]]

,power_presence,power_capacity
0,0,0.0
1,0,0.0
2,0,0.0
3,0,0.0
4,0,0.0
...,...,...
2382,0,0.0
2383,0,0.0
2384,0,0.0
2385,0,0.0


In [144]:
new_ambition

,Unnamed: 0,iso,entity_type,climactor_id,baseline_value,baseline_year,inv_year,target_year,target_value,target_record,GDAM_id,target_year_emission,annualised_ambition_from_baseline,annualised_ambition,2019_emissions_for_total_ambition,2030_emissions,absolute_ambition,geometry
0,6,ESP,City,00752000db04c515,11423.000,2005,2014.0,2030,40.00,00752000db04c515_2030,ESP.09.08.08222,6853.80000,0.013550,0.012225,9.436959e+03,6.853800e+03,2583.159046,"POLYGON ((1.79244 41.47367, 1.79245 41.47362, ..."
1,7,ITA,City,0083ef4ed71b869e,7739.785,2011,2011.0,2030,40.00,0083ef4ed71b869e_2030,ITA.19.86.003,4643.87100,0.017867,0.017867,6.700266e+03,4.643871e+03,2056.394823,"MULTIPOLYGON (((14.38951 37.59933, 14.39053 37..."
2,10,BEL,City,00887dae5af7b824,48759.200,2011,2020.0,2030,40.00,00887dae5af7b824_2030,BEL.2.1.3.22_1,29255.52000,0.017867,0.030747,4.221042e+04,2.925552e+04,12954.903325,"POLYGON ((5.02865 51.27665, 5.02886 51.27515, ..."
3,13,ITA,City,00b3f9e9c6778784,8965.400,2011,2011.0,2030,40.00,00b3f9e9c6778784_2030,ITA.19.84.007,5379.24000,0.017867,0.017867,7.761270e+03,5.379240e+03,2382.030269,"POLYGON ((13.09734 37.59018, 13.09771 37.5898,..."
4,17,ITA,City,00db591fd214bead,7876.169,2011,2011.0,2030,40.00,00db591fd214bead_2030,ITA.19.83.088,4725.70140,0.017867,0.017867,6.818332e+03,4.725701e+03,2092.630888,"POLYGON ((14.85797 38.12697, 14.85803 38.12668..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2382,9387,USA,City,ff4e338346fcdebb,2597305.150,2019,2019.0,2050,100.00,ff4e338346fcdebb_2050,USA.16.77.1600000US1921000,0.00000,0.022611,0.022611,2.597305e+06,2.019588e+06,577717.230931,"MULTIPOLYGON (((-93.70958 41.59183, -93.70951 ..."
2383,9393,ITA,City,ff8d794fe84c1e9d,22806.700,2011,2011.0,2050,40.42,ff8d794fe84c1e9d_2050,ITA.19.87.047,13588.23186,0.008742,0.008742,2.125960e+04,1.930223e+04,1957.372294,"POLYGON ((14.85188 37.60497, 14.85194 37.60491..."
2384,9400,BEL,City,ff904d7013b2688c,75769.000,2011,2020.0,2030,40.00,ff904d7013b2688c_2030,BEL.2.2.3.1_1,45461.40000,0.017867,0.028499,6.559258e+04,4.546140e+04,20131.176682,"POLYGON ((5.24222 50.87254, 5.24232 50.87112, ..."
2385,9403,ITA,City,ffa60dd141d5c943,83747.800,2010,2010.0,2030,40.00,ffa60dd141d5c943_2030,ITA.15.63.073,50248.68000,0.016966,0.016966,7.179440e+04,5.024868e+04,21545.717128,"POLYGON ((14.21384 40.95078, 14.21388 40.95064..."
